# 04 — Generation parameter tuning

36-config grid over `(temperature × top_p × num_predict)` on the first 20
queries of `golden_set_v1.jsonl`. Retrieval is cached once per query, so the
only moving variable is `Generator.generate()` via Ollama `/api/chat`.

Grid run by `scripts/tune_gen_params.py`; raw data in
`evaluation/results/gen_params_grid.json`. See `docs/design.md §4.4.1` for
the prose rationale.

**Chosen config:** `temperature=0.4, top_p=1.0, num_predict=800`.

In [ ]:
import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

grid_path = Path.cwd().parent / "evaluation" / "results" / "gen_params_grid.json"
with grid_path.open() as fh:
    data = json.load(fh)

rows = []
for r in data["rows"]:
    rows.append({
        "temperature": r["config"]["temperature"],
        "top_p": r["config"]["top_p"],
        "num_predict": r["config"]["num_predict"],
        "format_rate": r["format_rate"],
        "grounded_rate": r["grounded_rate"],
        "mean_confidence": r["mean_confidence"],
        "mean_citations": r["mean_citations"],
        "p95_latency_gen_ms": r["p95_latency_gen_ms"],
        "heuristic_score": r["heuristic_score"],
    })
df = pd.DataFrame(rows)
print(f"{len(df)} configs, {data['n_queries']} queries per config")
df.head()

## Top-10 configs (all orderings)

`fmt < 1.0` configs are shown but **must** be rejected — every one of them
clipped at least one answer on `num_predict=400`.

In [ ]:
top10 = df.sort_values(["heuristic_score", "grounded_rate", "p95_latency_gen_ms"],
                       ascending=[False, False, True]).head(10)
top10.reset_index(drop=True)

## fmt=1.0 configs only — the real ranking

In [ ]:
clean = df[df["format_rate"] >= 1.0].sort_values(
    ["heuristic_score", "grounded_rate", "p95_latency_gen_ms"],
    ascending=[False, False, True],
)
clean.head(10).reset_index(drop=True)

## Heatmap — heuristic score across (temperature × top_p)

Averaged over `num_predict=800` (the chosen budget).

In [ ]:
subset = df[df["num_predict"] == 800]
pivot = subset.pivot(index="temperature", columns="top_p", values="heuristic_score")
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(pivot.values, aspect="auto", cmap="viridis")
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels([f"{v:.2f}" for v in pivot.columns])
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels([f"{v:.2f}" for v in pivot.index])
ax.set_xlabel("top_p")
ax.set_ylabel("temperature")
ax.set_title("heuristic score (num_predict=800)")
for i, t in enumerate(pivot.index):
    for j, p in enumerate(pivot.columns):
        ax.text(j, i, f"{pivot.loc[t, p]:.3f}", ha="center", va="center", color="w")
fig.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

## num_predict trade-off — p95 latency vs citation richness

In [ ]:
grouped = df[df["format_rate"] >= 1.0].groupby("num_predict").agg(
    mean_score=("heuristic_score", "mean"),
    mean_cites=("mean_citations", "mean"),
    mean_p95_ms=("p95_latency_gen_ms", "mean"),
).reset_index()
grouped